# Riconoscere un gatto dagli esempi

Invece di scrivere regole, diamo al computer **migliaia di foto** etichettate "gatto" / "non gatto"
e partiamo da una rete già addestrata (MobileNetV2), aggiungendo solo l'ultimo pezzo.

Prima di iniziare: **Runtime → Cambia tipo di runtime → T4 GPU**.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import os
from PIL import Image

In [ ]:
# Scarichiamo le foto di cani e gatti pubblicate da Microsoft (circa 800 MB, un paio di minuti)
# e ne copiamo una parte nelle cartelle che il modello si aspetta. I cani fanno da "non gatto".
import random, shutil, urllib.request, zipfile

URL = 'https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip'
if not os.path.exists('PetImages'):
    urllib.request.urlretrieve(URL, 'gatti_cani.zip')
    with zipfile.ZipFile('gatti_cani.zip') as z:
        z.extractall('.')

def foto_valide(cartella, quante):
    nomi = sorted(os.listdir(cartella))
    random.Random(0).shuffle(nomi)
    buone = []
    for nome in nomi:
        percorso = os.path.join(cartella, nome)
        try:
            with Image.open(percorso) as img:
                img.verify()                     # alcune foto del dataset sono rovinate
            if Image.open(percorso).format == 'JPEG':
                buone.append(percorso)
        except Exception:
            pass
        if len(buone) == quante:
            return buone
    return buone

for origine, classe in [('PetImages/Cat', 'cats'), ('PetImages/Dog', 'not_cats')]:
    foto = foto_valide(origine, 1200)
    for parte, elenco in [('train', foto[:1000]), ('validation', foto[1000:])]:
        destinazione = f'data/{parte}/{classe}'
        os.makedirs(destinazione, exist_ok=True)
        for p in elenco:
            shutil.copy(p, destinazione)
print('Foto pronte')

In [ ]:
# Step 1: Load the pre-trained model
# By default, it expects images of size 224x224 pixels.
base_model = MobileNetV2(weights='imagenet', input_shape=(224, 224, 3), include_top=False) # Exclude the top layer
base_model.trainable = False  # Freeze the base model layers



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Step 2: Add custom layers for binary classification
x = base_model.output
x = GlobalAveragePooling2D()(x)  # Reduce the dimensions
#x = Dense(128, activation='relu')(x)  # Add a fully connected layer
x = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
predictions = Dense(1, activation='sigmoid')(x)  # Sigmoid for binary classification (cat vs. not-cat)

In [ ]:
# Step 3: Build the model
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Step 4: Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Step 5: Prepare the dataset
'''
data/
└── train/
    ├── cats/
    │   └── (cat images)
    └── not_cats/
        └── (not cat images)
└── validation/
    ├── cats/
    │   └── (cat images)
    └── not_cats/
        └── (not cat images)
'''

In [ ]:
train_dir = 'data/train'
validation_dir = 'data/validation'

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # MobileNetV2 vuole i pixel tra -1 e 1
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True,   # mescola: altrimenti ogni lotto contiene solo gatti o solo non gatti
    classes=['cats', 'not_cats']  # Explicitly define classes
)

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False, # maintain the original order of images
    classes=['cats', 'not_cats']  # Explicitly define classes
)

print(train_generator.class_indices)


x,y = next(train_generator)
for i in range(0,1):
    image = (x[i] + 1) / 2   # da [-1, 1] a [0, 1] per mostrarla
    plt.imshow(image)
    plt.show()
    print(y[i])

In [ ]:
# Step 6: Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=5,
    verbose=1  # Set verbose to 1 to display progress and metrics
)

In [ ]:
# Access training loss and accuracy
training_loss = history.history['loss']
training_accuracy = history.history['accuracy']

# Access validation loss and accuracy
validation_loss = history.history['val_loss']
validation_accuracy = history.history['val_accuracy']

# Print the values
print("Training Loss:", training_loss)
print("Training Accuracy:", training_accuracy)
print("Validation Loss:", validation_loss)
print("Validation Accuracy:", validation_accuracy)

Training Loss: [3.600135326385498, 2.241852045059204, 2.966675043106079, 2.3356075286865234, 2.265845537185669, 1.8201854228973389, 1.909805417060852, 1.8063991069793701, 1.6399686336517334, 1.8443626165390015]
Training Accuracy: [0.6578947305679321, 1.0, 0.5751879811286926, 1.0, 0.8308270573616028, 1.0, 0.8909774422645569, 0.9375, 0.9624060392379761, 0.84375]
Validation Loss: [2.641963481903076, 3.961371898651123, 2.2857446670532227, 2.512908458709717, 2.0531721115112305, 1.9770004749298096, 1.7742900848388672, 1.9263408184051514, 1.6017346382141113, 1.8488693237304688]
Validation Accuracy: [0.6458333134651184, 0.0, 0.8368055820465088, 0.699999988079071, 0.8993055820465088, 1.0, 0.9375, 1.0, 0.9618055820465088, 0.8999999761581421]


In [ ]:
# Step 7: Save the model
model.save('cat_recognizer_model.keras')

In [ ]:
# Step 8: Test with an image
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = preprocess_input(img_to_array(img))
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    prediction = model.predict(img_array)
    p = prediction[0][0]   # probabilità di "not_cats": la classe 0 è cats, la 1 è not_cats
    return f"Non gatto ({p:.0%})" if p > 0.5 else f"Gatto ({1 - p:.0%})"


In [ ]:
from google.colab import files

# Test the model
uploaded = files.upload()
image_path = next(iter(uploaded))

print(predict_image(image_path))